# S02 — Control Flow, Functions & the VS Code Debugger

**Python for Optics** · Master LaScala  
Session 2 of 10 · 2 hours

---

## Session roadmap

| # | Topic | Approx. time |
|---|-------|--------------|
| 1 | Conditionals | 15 min |
| 2 | Loops: `for` and `while` | 20 min |
| 3 | Comprehensions | 10 min |
| 4 | Functions: definition, docstrings, scope | 25 min |
| 5 | Lambda functions | 10 min |
| 6 | VS Code Debugger | 20 min |
| — | Exercises | 60 min |

---

**What you need from S01:** Python types, f-strings, lists, dicts, the `math` module, a working VS Code + Jupyter setup.

**What you will be able to do after S02:** Write functions that express physical models cleanly, loop over parameter spaces, and step through any misbehaving code with the debugger instead of guessing.

---
## 1 · Conditionals

Python uses `if / elif / else`. Indentation (4 spaces) defines the block — there are no braces.

In [ ]:
# Classify a laser by its average power
P_avg = 0.85  # W

if P_avg < 1e-3:
    regime = "Class 1 (eye-safe)"
elif P_avg < 0.5:
    regime = "Class 3B (medium power)"
elif P_avg < 5.0:
    regime = "Class 4 (high power — beam stop required)"
else:
    regime = "Industrial / research grade"

print(f"P_avg = {P_avg:.3f} W  →  {regime}")

### Comparison and logical operators

| Operator | Meaning |
|----------|---------|
| `==` `!=` | equal / not equal |
| `<` `<=` `>` `>=` | numeric comparison |
| `and` `or` `not` | logical combination |
| `in` | membership test |
| `is` | identity (not equality) |

In [ ]:
wavelength_nm = 1030  # Yb:KGW common output
rep_rate_MHz  = 76

is_nir     = 780 < wavelength_nm < 2000
is_highRep = rep_rate_MHz > 50

print(f"NIR: {is_nir},  High-rep: {is_highRep}")
print(f"Both: {is_nir and is_highRep}")

### `==` vs `is`  

Each time you create a python object (like list, dictionaries., etc.) the interpreter allocates a memory space for it. The name used for the object contains the address to the object in memory (is a pointer).

In [ ]:
mylist1= [1, 2, 3]  # mylist1 is the pointer to the mekory loocation where [1,2,3] is stored

print(mylist1)  # [1,2,3]

mylist2 = mylist1   # mylist2 contains exactly the same address as maylist1. So both are poibnting to the same memory location

print(mylist2)  # [1,2,3]

# mylist1 and mylist 2 point to the same object, therefore

mylist1[1]=10
print(mylist1)  # [1,10,3]
print(mylist2)  # [1,10,3]

# Each object has an identificator 
print(id(mylist1))   
print(id(mylist2))   # samne id as mylist1



Use `==` to compare *values*. Use `is` to test if the object points by the names are the same 

In [ ]:
mylist1= [1, 2, 3]
mylist2=mylist1

print(mylist1==mylist2)     # True
print(mylist1 is mylist2)   #True

mylist1= [1, 2, 3]
mylist2= [1,2,3]    # now my list2 is defined as anew object, stored in a different place in memory

print(mylist1==mylist2)     # True
print(mylist1 is mylist2)   # False


> Sometimes is useful to prepare an empty pointer (a name that points to nothing) using `None`

In [ ]:

a = None

print(a) # None

print(id(a)) # Some id

print (a is None) # True


### Ternary (inline) conditional

In [ ]:
pulse_duration_fs = 35
label = "ultrashort" if pulse_duration_fs < 100 else "short"
print(f"{pulse_duration_fs} fs  →  {label} pulse")

---
## 2 · Loops

### 2.1 · `for` loop

`for` iterates over any *iterable* — a list, a range, a string, a dict, …

In [ ]:
# Common laser wavelengths (nm)
wavelengths = [355, 515, 532, 1030, 1064]

print(f"{'λ (nm)':>8}  {'ν (THz)':>10}")
print("-" * 22)
for lam_nm in wavelengths:
    c = 3e8          # m/s
    nu = c / (lam_nm * 1e-9) / 1e12   # THz
    print(f"{lam_nm:>8d}  {nu:>10.2f}")

In [ ]:
# range(start, stop, step) — stop is excluded
for i in range(0, 5):
    print(i, end=" ")
print()

# enumerate gives index AND value
materials = ["BBO", "KDP", "LBO", "PPLN"]
for idx, mat in enumerate(materials, start=1):
    print(f"  {idx}. {mat}")

### `zip` — iterate two sequences in parallel

In [ ]:
crystals   = ["BBO",  "KDP",  "LBO"]
phase_match = [22.8,  41.0,   11.4]    # degrees (example values)

for crystal, theta in zip(crystals, phase_match):
    print(f"  {crystal}: θ_PM = {theta:.1f}°")

### 2.2 · `while` loop

Use `while` when the number of iterations is not known in advance — typical for convergence loops.

In [ ]:
# Iterative bisection — find the beam waist position where w(z) crosses a threshold
# w(z) = w0 * sqrt(1 + (z/z_R)^2)

# Problea: Find the position z of the point in the beam where w(z)=w_test
import math

w0    = 50e-6   # m   beam waist
lam   = 1030e-9 # m
z_R   = math.pi * w0**2 / lam   # Rayleigh range  7.625 mm
w_test = 70.71e-6   # m   test  is should be when z=z_R, since w(z_R) = w0 * sqrt(2)=50e-6*sqrt(2)=70.71e-6

z_lo, z_hi = 0.0, z_R * 10
tol = 1e-6  # m
iterations = 0

while (z_hi - z_lo) > tol:
    z_mid = (z_lo + z_hi) / 2
    w_mid = w0 * math.sqrt(1 + (z_mid / z_R)**2)
    if w_mid < w_test:
        z_lo = z_mid
    else:
        z_hi = z_mid
    iterations += 1

print(f"z where w(z) = {w_test*1e6:.0f} µm : {z_mid*1e3:.3f} mm (z_R found!)")
print(f"Converged in {iterations} iterations")

> **⚠️ Common Pitfall — Infinite loops**  
> Always make sure the loop condition can eventually become `False`.  
> Add a safety counter if needed: `if iterations > 1000: break`

### `break`, `continue`, `else` on loops

In [ ]:
energies_uJ = [0.5, 1.2, 3.8, 0.9, 15.0, 2.1]   # pulse energy per shot
threshold   = 10.0  # µJ — damage threshold

for i, E in enumerate(energies_uJ):
    if E > threshold:
        print(f"ALARM: shot {i} exceeded threshold ({E:.1f} µJ). Stopping.")
        break
    print(f"Shot {i}: {E:.1f} µJ — OK")
else:
    # 'else' on a loop runs only if the loop completed WITHOUT a break
    print("All shots within tolerance.")

---
## 3 · Comprehensions

A compact syntax to build lists (or dicts/sets) from existing sequences. Preferred over explicit `for` + `append` in Python.

In [23]:
import math

# List comprehension — frequency (THz) for each wavelength (nm)
wavelengths_nm = [355, 515, 532, 800, 1030, 1064]
c = 3e8  # m/s

frequencies_THz = [c / (lam * 1e-9) / 1e12 for lam in wavelengths_nm]
print("Frequencies (THz):", [f"{nu:.1f}" for nu in frequencies_THz])

Frequencies (THz): ['845.1', '582.5', '563.9', '375.0', '291.3', '282.0']


In [ ]:
# With a filter — only wavelengths in the NIR window
nir = [lam for lam in wavelengths_nm if 780 < lam < 2000 ]
print("NIR wavelengths:", nir)

# Dict comprehension — λ → ν mapping
lam_to_nu = {lam: c / (lam * 1e-9) / 1e12 for lam in wavelengths_nm}
print("λ=1030 nm →", f"{lam_to_nu[1030]:.2f} THz")

> **💡 Pro Tip**  
> Keep comprehensions to a single logical operation. If you need two nested loops etc, write a regular `for` loop — readability matters more than brevity.

> **💡 Pro Tip** 
> You can add `else`to a conditional list comprehension, but you have to chage the structure
> - Without `else``: the conditional comes __after__ the container
> ```python
> nir = [lam for lam in wavelengths_nm if 780 < lam < 2000 ]
> ```
> - With `else``: the conditional comes __before__ the container
> ```python
> nir = [lam  if 780 < lam < 2000 else None for lam in wavelengths_nm]
> ```

In [26]:
nir = [lam for lam in wavelengths_nm if 780 < lam < 2000]
print(nir)  # [800, 1030, 1064]

nir = [lam  if 780 < lam < 2000 else None for lam in wavelengths_nm]
print(nir)  # [None, None, None, 800, 1030, 1064]

[800, 1030, 1064]
[None, None, None, 800, 1030, 1064]


---
## 4 · Functions

Functions are the basic unit of reusable, testable, readable code. In physics simulations, every formula worth using more than once deserves a function.

### 4.1 · Basic definition and NumPy-style docstrings

In [27]:
import math

def gaussian_beam_waist(z, w0, wavelength):
    """
    Beam radius of a Gaussian beam at propagation distance z.

    Parameters
    ----------
    z : float
        Propagation distance from the beam waist [m].
    w0 : float
        Beam waist radius (1/e² intensity) [m].
    wavelength : float
        Laser wavelength [m].

    Returns
    -------
    float
        Beam radius w(z) [m].
    """
    z_R = math.pi * w0**2 / wavelength
    return w0 * math.sqrt(1 + (z / z_R)**2)


# Call the function
w0  = 50e-6    # 50 µm
lam = 1030e-9  # 1030 nm

for z_mm in [0, 50, 100, 200]:
    w = gaussian_beam_waist(z_mm * 1e-3, w0, lam)
    print(f"z = {z_mm:>4} mm  →  w(z) = {w*1e6:.1f} µm")

z =    0 mm  →  w(z) = 50.0 µm
z =   50 mm  →  w(z) = 331.6 µm
z =  100 mm  →  w(z) = 657.6 µm
z =  200 mm  →  w(z) = 1312.4 µm


> **🔬 Physics Insight — Why docstrings and why the NumPy docstring convention?**  
> NumPy-style docstrings (Parameters / Returns / Notes sections) are the standard in the scientific Python ecosystem. Tools like Jupyter show them on hover, and Sphinx can auto-generate HTML documentation from them. Start writing them now — the habit pays off the moment someone else (or future-you) reads your code.
>
> At anytime you can retrieve the docstring info unsing `help`

In [28]:
help(gaussian_beam_waist)

Help on function gaussian_beam_waist in module __main__:

gaussian_beam_waist(z, w0, wavelength)
    Beam radius of a Gaussian beam at propagation distance z.
    
    Parameters
    ----------
    z : float
        Propagation distance from the beam waist [m].
    w0 : float
        Beam waist radius (1/e² intensity) [m].
    wavelength : float
        Laser wavelength [m].
    
    Returns
    -------
    float
        Beam radius w(z) [m].



### 4.2 · Default arguments and keyword arguments

In [29]:
def rayleigh_range(w0, wavelength=1030e-9):
    """
    Rayleigh range of a Gaussian beam.

    Parameters
    ----------
    w0 : float
        Beam waist radius [m].
    wavelength : float, optional
        Laser wavelength [m]. Default is 1030 nm (Yb-based laser).

    Returns
    -------
    float
        Rayleigh range [m].
    """
    return math.pi * w0**2 / wavelength


zR_yb  = rayleigh_range(50e-6)            # uses default λ=1030 nm
zR_ti  = rayleigh_range(50e-6, 800e-9)    # Ti:Sapphire
zR_ti2 = rayleigh_range(w0=50e-6, wavelength=800e-9)  # keyword syntax

print(f"z_R (Yb, 1030 nm): {zR_yb*1e3:.2f} mm")
print(f"z_R (Ti:Sa, 800 nm): {zR_ti*1e3:.2f} mm")

z_R (Yb, 1030 nm): 7.63 mm
z_R (Ti:Sa, 800 nm): 9.82 mm


> **⚠️ Common Pitfall — Mutable default arguments**  
> Never use a mutable object (list, dict) as a default argument.  
> ```python
> def bad(data=[]):    # DON'T — the list is shared across all calls!
>     data.append(1)
>     return data
> ```  
> Use `None` as default and create the object inside the function body.

In [35]:
def bad(data=[]):    # DON'T — the list is shared across all calls!
    data.append(1)
    return data

def good(data=None):    # USE Nopne instead
    if data is None:
        data=[]
    data.append(1)
    return data

print('----- bad') 
data=bad()
print(data)  

data=bad()
print(data)


print('----- good') 

data=good()
print(data)  

data=good()
print(data)


----- bad
[1]
[1, 1]
----- good
[1]
[1]


### 4.3 · Multiple return values

In [36]:
def pulse_parameters(E_J, tau_s, rep_rate_Hz):
    """
    Compute key parameters of a pulsed laser from measurable quantities.

    Parameters
    ----------
    E_J : float
        Pulse energy [J].
    tau_s : float
        Pulse duration (FWHM) [s].
    rep_rate_Hz : float
        Repetition rate [Hz].

    Returns
    -------
    P_peak : float
        Peak power [W] (assumes Gaussian pulse shape, factor 0.94).
    P_avg : float
        Average power [W].
    duty_cycle : float
        Duty cycle (dimensionless).
    """
    P_peak     = 0.94 * E_J / tau_s           # Gaussian pulse approximation
    P_avg      = E_J * rep_rate_Hz
    duty_cycle = tau_s * rep_rate_Hz
    return P_peak, P_avg, duty_cycle


# Yb:KGW oscillator — typical values
E       = 50e-9    # 50 nJ
tau     = 300e-15  # 300 fs
f_rep   = 76e6     # 76 MHz

P_peak, P_avg, dc = pulse_parameters(E, tau, f_rep)
print(f"Peak power    : {P_peak/1e3:.2f} kW")
print(f"Average power : {P_avg:.3f} W")
print(f"Duty cycle    : {dc:.2e}")

Peak power    : 156.67 kW
Average power : 3.800 W
Duty cycle    : 2.28e-05


### 4.4 · `*args` and `**kwargs`

In [37]:
def print_laser_report(laser_name, *specs, **settings):
    """
    Print a formatted laser report.
    *specs   : positional extra info (wavelength, power, ...)
    **settings : key-value optional settings
    """
    print(f"=== {laser_name} ===")
    for s in specs:
        print(f"  - {s}")
    for key, val in settings.items():
        print(f"  {key}: {val}")


print_laser_report(
    "Pharos (Light Conversion)",
    "λ = 1030 nm",
    "τ = 190–10000 fs",
    f_rep="1–200 kHz",
    P_avg="up to 20 W",
    M2="< 1.3"
)

=== Pharos (Light Conversion) ===
  - λ = 1030 nm
  - τ = 190–10000 fs
  f_rep: 1–200 kHz
  P_avg: up to 20 W
  M2: < 1.3


### 4.5 · Scope — LEGB rule

Python looks up names in this order: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.

In [39]:
c = 3e8  # Global constant (speed of light, m/s)

def wavelength_to_frequency(lam_nm):
    # 'c' is found in Global scope — fine for physical constants
    return c / (lam_nm * 1e-9)   # Hz

def bad_practice():
    global c           # DON'T modify globals inside functions
    c = 2.998e8        # now c is changed for EVERYONE

# Prefer passing constants as arguments:
def wavelength_to_frequency_v2(lam_nm, c=c):
    return c / (lam_nm * 1e-9)

print(f"ν at 1030 nm = {wavelength_to_frequency(1030)/1e12:.3f} THz")

ν at 1030 nm = 291.262 THz


> **💡 Pro Tip — Physical constants**  
> For serious code, use `scipy.constants` instead of typing out constants by hand.  
> ```python
> from scipy.constants import c, h, hbar, epsilon_0
> ```  
> All values are CODATA-recommended and consistently named.
>
> If you   are not sure about the unit conventions used in scipy.constants, import the function `find` and the dictionary `physical constants``
> ```python
> from scipy.constants import c, h, hbar, epsilon_0, find, physical_constants
> 
> print(find('light'))  # ['speed of light in vacuum']
> value, unit, uncertainty =physical_constants['speed of light in vacuum']
> print(f"Value:       {value}") # Value:       299792458.0
> print(f"Unit:        {unit}")  # Unit:        m s^-1
> print(f"Uncertainty: {uncertainty}") # Uncertainty: 0.0
> ``` 

---
## 5 · Lambda functions

Anonymous single-expression functions. Useful for short transformations, especially when passing a function as an argument.

In [ ]:
# Lambda syntax: lambda arguments: expression
to_THz = lambda lam_nm: 3e8 / (lam_nm * 1e-9) / 1e12

print(f"1030 nm → {to_THz(1030):.2f} THz")
print(f" 800 nm → {to_THz(800):.2f} THz")

# Most useful: as a key function for sorting
lasers = [
    {"name": "Yb:KGW",   "wavelength": 1030, "power_W": 2.0},
    {"name": "Ti:Sa",     "wavelength":  800, "power_W": 1.5},
    {"name": "Nd:YAG",   "wavelength": 1064, "power_W": 5.0},
    {"name": "Er:fiber",  "wavelength": 1550, "power_W": 0.5},
]

sorted_by_power = sorted(lasers, key=lambda laser: laser["power_W"], reverse=True)
for L in sorted_by_power:
    print(f"  {L['name']:12s}  λ={L['wavelength']} nm  P={L['power_W']} W")

> **⚠️ Common Pitfall**  
> Lambdas are fine for one-liners passed to `sorted`, `map`, `filter`, or as callbacks.  
> If you find yourself writing a complex lambda, write a named function instead — you'll thank yourself when debugging.

---
## 6 · VS Code Debugger

The debugger lets you pause execution and inspect the state of every variable at any line. It is the correct tool for understanding **why** code misbehaves — far more efficient than `print()` statements.

### 6.1 · Activating Error Lens and indent-rainbow

If you haven't installed these extensions yet:
1. Open the Extensions sidebar (`Ctrl+Shift+X` / `Cmd+Shift+X`)
2. Search and install:
   - **Error Lens** (Alexander) — shows errors and warnings inline on the line they occur
   - **indent-rainbow** (oderwat) — colors indentation levels, invaluable in nested Python code

---

### 6.2 · Debugging a Python script (`.py` file)

**Key actions:**

| Action | Shortcut |
|--------|----------|
| Set/remove breakpoint | Click left of line number, or `F9` |
| Start debugging | `F5` (opens Run & Debug panel first time) |
| Step over (next line, don't enter function) | `F10` |
| Step into (enter function call) | `F11` |
| Step out (finish current function, return) | `Shift+F11` |
| Continue to next breakpoint | `F5` |
| Stop | `Shift+F5` |

**Panels to watch:**
- **Variables** — all local and global variables at the current line
- **Watch** — add any expression to evaluate continuously (e.g. `z_R * 1e3`)
- **Call stack** — shows which functions called which, useful in nested code
- **Debug Console** — run arbitrary Python expressions at the current breakpoint

---

### 6.3 · Debugging a Jupyter notebook cell

1. Open the notebook in VS Code.
2. Click to the left of a cell's line numbers to set a breakpoint (a red dot appears).
3. Click the **Debug Cell** button (bug icon) that appears above the cell, or right-click → **Debug Cell**.
4. The same debugger panels open as for scripts.

> **💡 Pro Tip**  
> The Debug Console is particularly useful in notebooks: while paused at a breakpoint, type any expression to inspect intermediate results — for example `w0 * 1e6` to see the beam waist in µm even if you forgot to print it.

---

### 6.4 · A deliberately broken function to debug

The function below has a subtle bug. Use the debugger (not print statements!) to find it.

In [5]:
def peak_irradiance(E_J, tau_s, w0_m):
    """
    Peak irradiance at the beam waist of a Gaussian pulse.

    I_peak = 2 * E / (pi * w0^2 * tau)

    Parameters
    ----------
    E_J   : float  Pulse energy [J]
    tau_s : float  Pulse duration FWHM [s]
    w0_m  : float  Beam waist radius [m]

    Returns
    -------
    float
        Peak irradiance [W/m²]
    """
    import math
    # BUG: can you spot it before running the debugger?
    I_peak = 2 * E_J / (math.pi * w0_m * tau_s)
    return I_peak


# Test: 1 µJ pulse, 300 fs, 50 µm waist
# Expected ~  2 * 1e-6 / (pi * (50e-6)^2 * 300e-15)  ≈  8.5e11 W/m²
I = peak_irradiance(1e-6, 300e-15, 50e-6)
print(f"I_peak = {I:.3e} W/m²")
print(f"I_peak = {I/1e4:.3e} W/cm²")

I_peak = 4.244e+10 W/m²
I_peak = 4.244e+06 W/cm²


<details>
<summary>⚡ Try It — Debugging steps (click to reveal)</summary>

1. Set a breakpoint on the `I_peak = ...` line.
2. Click **Debug Cell**.
3. When paused, open the Debug Console and evaluate: `math.pi * w0_m * tau_s` vs `math.pi * w0_m**2 * tau_s`.
4. The bug: `w0_m` should be `w0_m**2`. The beam area is `π·w₀²`, not `π·w₀`.
5. Fix the formula and rerun — you should get ≈ 8.49 × 10¹¹ W/m².

</details>

---
## 7 · Session summary

| Concept | Key point |
|---------|----------|
| `if/elif/else` | Use chained `elif` for multi-way branching; avoid deeply nested ifs |
| `for` loop | Prefer over `while` when the iterable is known; use `enumerate`, `zip` |
| `while` loop | For convergence loops; always ensure termination |
| Comprehensions | Clean and Pythonic, but only for simple one-liners |
| Functions | Named, documented with NumPy-style docstrings, single responsibility |
| Default args | Use `None` as default for mutable objects |
| Scope | Avoid `global`; pass constants as arguments |
| Lambda | Fine for short key/transform functions; use named functions for anything complex |
| Debugger | Breakpoints → step → watch → Debug Console. Never guess what a variable holds. |

**Next session (S03):** Git and GitHub — we will start version-controlling everything we write from here on.